# 03 - frozen encoders: extract, probe, save (one loop)

**GPU runtime.** For every (encoder, normalisation) in `RUNS`: extract frozen
patch features for train + val, train the linear zone probe, score it on the
validation glacier SI, save `probe_zones__<run>.csv` to Drive, then delete the
run's features so the disk never fills up.

- Runs whose results CSV is already on Drive are **skipped**: after a crash or a
  runtime reset, just run this notebook again.
- One failing model does not stop the others.
- Test glaciers (COL, Mapple) are never touched here.

| encoder | pretraining | normalisation |
|---|---|---|
| `dinov3-l-sat` | **optical satellite (SAT-493M)** -- the DINOv3 of the study since 2026-09-25 | `per_image`, `caffe` |
| `dinov3-l-photo` | photos (LVD-1689M); same ViT-L/16, kept as the matched pair, already run | `per_image`, `caffe` |
| `cradio-v4-h` | photos, multi-teacher | `per_image`, `caffe` |
| `terramind-v1-base-s1` | Sentinel-1 radar + optical | `per_image`, `caffe` |
| `saratrx-v1` | SAR target chips | `unit` (its pretraining input, pixel/255) |

Measured in the saved run below: 6.7 and 5.5 min per run (the two `dinov3-l-sat`
runs); the first loop took 43 min (docs/results.md, Run times). Then open
`04_compare` for the tables.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, data ---
import os, shutil
from pathlib import Path
import torch
from sartransfer.env import load_env, require
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)

DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
CAFFE, FEAT = Path("/content/caffe"), Path("/content/features")
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), CAFFE)
DATA = inv.find_data_root(CAFFE)
df = inv.scan(DATA, subdirs=("sar_images",))
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: RUN EVERYTHING ---
import time
from sartransfer.runs import run_encoder

RUNS = [                                    # (encoder, normalisation)
    ("dinov3-l-photo", "per_image"),        # skipped in the saved run (already on Drive)
    ("dinov3-l-photo", "caffe"),            # skipped in the saved run (already on Drive)
    ("dinov3-l-sat", "per_image"),          # SAT-493M twin of the photo model (added 2026-09-25)
    ("dinov3-l-sat", "caffe"),
    ("cradio-v4-h", "per_image"),
    ("cradio-v4-h", "caffe"),
    ("terramind-v1-base-s1", "per_image"),
    ("terramind-v1-base-s1", "caffe"),
    ("saratrx-v1", "unit"),
    # check: TerraMind was pretrained on 224 px; is the low 512 px score our tiling?
    ("terramind-v1-base-s1", "caffe", 224),
]

t0, status = time.perf_counter(), {}
for encoder, norm, *rest in RUNS:          # optional third item: tile size (default 512)
    tile = rest[0] if rest else 512
    key = f"{encoder} | {norm} | {tile}px"
    try:
        r = run_encoder(encoder, norm, df, DATA, FEAT, RES,
                        token=os.environ["HF_TOKEN"], tile=tile)
        status[key] = "skipped (already saved)" if r is None else "done"
    except Exception as e:
        status[key] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(status[key])

print(f"\n===== finished in {(time.perf_counter() - t0) / 60:.0f} min =====")
for k, v in status.items():
    print(f"  {k:<44} {v}")

skip dinov3-l-photo__per_image__tile512: results already on Drive
skip dinov3-l-photo__caffe__tile512: results already on Drive

===== dinov3-l-sat__per_image__tile512 =====


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  50/559  0.4 min elapsed, 0.50 s/image, ETA 4.2 min
  100/559  0.7 min elapsed, 0.42 s/image, ETA 3.2 min
  150/559  0.9 min elapsed, 0.35 s/image, ETA 2.4 min
  200/559  1.4 min elapsed, 0.41 s/image, ETA 2.4 min
  250/559  2.0 min elapsed, 0.48 s/image, ETA 2.4 min
  300/559  2.6 min elapsed, 0.52 s/image, ETA 2.2 min
  350/559  3.2 min elapsed, 0.55 s/image, ETA 1.9 min
  400/559  3.5 min elapsed, 0.53 s/image, ETA 1.4 min
  450/559  3.7 min elapsed, 0.50 s/image, ETA 0.9 min
  500/559  3.8 min elapsed, 0.46 s/image, ETA 0.5 min
  550/559  4.2 min elapsed, 0.45 s/image, ETA 0.1 min
  559/559  4.3 min elapsed, 0.46 s/image, ETA 0.0 min
extracted: FrozenEncoder(dinov3-l-sat, kind=dinov3, patch=16, dim=1024, n_prefix=5) in 4.8 min
subsampling train: 9,528,284 valid patches -> ~2,000,000 (21.0% per file)
train: X (2001574, 1024) 8.20 GB, y (2001574,)
mIoU 0.5716 (balanced, SI) vs majority b

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  50/559  0.3 min elapsed, 0.41 s/image, ETA 3.5 min
  100/559  0.6 min elapsed, 0.36 s/image, ETA 2.7 min
  150/559  0.7 min elapsed, 0.30 s/image, ETA 2.0 min
  200/559  1.2 min elapsed, 0.35 s/image, ETA 2.1 min
  250/559  1.7 min elapsed, 0.40 s/image, ETA 2.1 min
  300/559  2.2 min elapsed, 0.44 s/image, ETA 1.9 min
  350/559  2.7 min elapsed, 0.47 s/image, ETA 1.6 min
  400/559  3.0 min elapsed, 0.45 s/image, ETA 1.2 min
  450/559  3.1 min elapsed, 0.42 s/image, ETA 0.8 min
  500/559  3.3 min elapsed, 0.39 s/image, ETA 0.4 min
  550/559  3.5 min elapsed, 0.39 s/image, ETA 0.1 min
  559/559  3.6 min elapsed, 0.39 s/image, ETA 0.0 min
extracted: FrozenEncoder(dinov3-l-sat, kind=dinov3, patch=16, dim=1024, n_prefix=5) in 3.7 min
subsampling train: 9,528,284 valid patches -> ~2,000,000 (21.0% per file)
train: X (2001574, 1024) 8.20 GB, y (2001574,)
mIoU 0.5714 (balanced, SI) vs majority b